# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier,
    GradientBoostingClassifier, AdaBoostClassifier
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


**Data preparation (order of the steps)**

The preprocessing has to follow this order, because every step needs the output of the previous one:

1. Define the target `y` and the feature matrix `X` (with the feature engineering that does not depend on the data).
2. **Train/test split.**
3. Fill missing values and encode the categorical variables, using only statistics from the training set.
4. Feature scaling.
5. Feature selection.

All the ensemble models below are trained and tested on the same scaled and selected features.

In [3]:
# 1. Target and feature matrix
y = spaceship["Transported"].astype(int)          # 1 = transported, 0 = not transported

X = spaceship.drop(columns=["Transported", "PassengerId", "Name"])   # ids and names carry no signal

# Feature engineering
# - Cabin has the form deck/num/side: keep the deck and the side
cabin = X["Cabin"].str.split("/", expand=True)
X["Deck"] = cabin[0]
X["Side"] = cabin[2]
X = X.drop(columns="Cabin")

# - Missing spending means no spending (passengers in cryosleep cannot spend); add the total spent
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
X[spend_cols] = X[spend_cols].fillna(0)
X["TotalSpend"] = X[spend_cols].sum(axis=1)

print("X shape:", X.shape, "| y shape:", y.shape)
X.head()

X shape: (8693, 13) | y shape: (8693,)


,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Deck,Side,TotalSpend
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,B,P,0.0
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,F,S,736.0
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,A,S,10383.0
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,A,S,5176.0
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,F,S,1091.0


**Perform Train Test Split**

In [4]:
# 2. Train/test split (stratified, so both sets keep the same share of transported passengers)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (6954, 13)
X_test shape: (1739, 13)
y_train shape: (6954,)
y_test shape: (1739,)


In [5]:
# 3. Missing values and categorical variables (statistics computed on the training set only)
num_cols = ["Age"] + spend_cols + ["TotalSpend"]
cat_cols = ["HomePlanet", "CryoSleep", "Destination", "VIP", "Deck", "Side"]

X_train[num_cols] = X_train[num_cols].fillna(X_train[num_cols].median())
X_test[num_cols] = X_test[num_cols].fillna(X_train[num_cols].median())

modes = X_train[cat_cols].mode().iloc[0]
X_train[cat_cols] = X_train[cat_cols].fillna(modes)
X_test[cat_cols] = X_test[cat_cols].fillna(modes)

# One-hot encoding; the test set gets exactly the same columns as the training set
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True, dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Missing values left:", X_train.isnull().sum().sum(), "(train),", X_test.isnull().sum().sum(), "(test)")
print("Columns:", X_train.columns.tolist())

Missing values left: 0 (train), 0 (test)
Columns: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpend', 'HomePlanet_Europa', 'HomePlanet_Mars', 'CryoSleep_True', 'Destination_PSO J318.5-22', 'Destination_TRAPPIST-1e', 'VIP_True', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T', 'Side_S']


In [6]:
# 4. Feature Scaling (fit on the training set only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

# 5. Feature Selection: the 10 best features according to the ANOVA F-test
selector = SelectKBest(score_func=f_classif, k=10)
selector.fit(X_train_scaled, y_train)
selected_features = X_train_scaled.columns[selector.get_support()]

# These are the matrices used by every model below
X_train_selected = X_train_scaled[selected_features]
X_test_selected = X_test_scaled[selected_features]

print("Selected Features:", selected_features.tolist())
print("Shapes:", X_train_selected.shape, X_test_selected.shape)

Selected Features: ['RoomService', 'Spa', 'VRDeck', 'TotalSpend', 'HomePlanet_Europa', 'CryoSleep_True', 'Deck_B', 'Deck_C', 'Deck_F', 'Side_S']
Shapes: (6954, 10) (1739, 10)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

In [7]:
results = {}   # metrics of every model, to compare them at the end

def evaluate(name, model):
    """Predict with a fitted model on the selected test features, print the metrics and store them."""
    y_pred = model.predict(X_test_selected)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name} Accuracy: {accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    results[name] = {
        "accuracy": accuracy,
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
    }

In [8]:
# Reference models (simple, non-ensemble) trained on the same features, to see whether ensembles really do better
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_selected, y_train)
evaluate("Logistic Regression", log_reg)

print("\n" + "=" * 60 + "\n")

tree_clf = DecisionTreeClassifier(random_state=42)
tree_clf.fit(X_train_selected, y_train)
evaluate("Decision Tree (single)", tree_clf)

Logistic Regression Accuracy: 0.7907

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.78      0.79       863
           1       0.79      0.80      0.79       876

    accuracy                           0.79      1739
   macro avg       0.79      0.79      0.79      1739
weighted avg       0.79      0.79      0.79      1739

Confusion Matrix:
[[674 189]
 [175 701]]


Decision Tree (single) Accuracy: 0.7435

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.67      0.72       863
           1       0.71      0.82      0.76       876

    accuracy                           0.74      1739
   macro avg       0.75      0.74      0.74      1739
weighted avg       0.75      0.74      0.74      1739

Confusion Matrix:
[[575 288]
 [158 718]]


- Bagging and Pasting

In [9]:
# Bagging: every tree is trained on a random sample drawn WITH replacement (bootstrap=True)
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(random_state=42),
    n_estimators=100, max_samples=0.8, bootstrap=True,
    n_jobs=-1, random_state=42
)
bag_clf.fit(X_train_selected, y_train)
evaluate("Bagging", bag_clf)

print("\n" + "=" * 60 + "\n")

# Pasting: the same, but the samples are drawn WITHOUT replacement (bootstrap=False)
paste_clf = BaggingClassifier(
    DecisionTreeClassifier(random_state=42),
    n_estimators=100, max_samples=0.8, bootstrap=False,
    n_jobs=-1, random_state=42
)
paste_clf.fit(X_train_selected, y_train)
evaluate("Pasting", paste_clf)

Bagging Accuracy: 0.7861

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.76      0.78       863
           1       0.78      0.81      0.79       876

    accuracy                           0.79      1739
   macro avg       0.79      0.79      0.79      1739
weighted avg       0.79      0.79      0.79      1739

Confusion Matrix:
[[659 204]
 [168 708]]




Pasting Accuracy: 0.7746

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.75      0.77       863
           1       0.76      0.80      0.78       876

    accuracy                           0.77      1739
   macro avg       0.78      0.77      0.77      1739
weighted avg       0.78      0.77      0.77      1739

Confusion Matrix:
[[645 218]
 [174 702]]


- Random Forests

In [10]:
forest_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
forest_clf.fit(X_train_selected, y_train)
evaluate("Random Forest", forest_clf)

Random Forest Accuracy: 0.7913

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.78      0.79       863
           1       0.79      0.80      0.80       876

    accuracy                           0.79      1739
   macro avg       0.79      0.79      0.79      1739
weighted avg       0.79      0.79      0.79      1739

Confusion Matrix:
[[672 191]
 [172 704]]


- Gradient Boosting

In [11]:
gb_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb_clf.fit(X_train_selected, y_train)
evaluate("Gradient Boosting", gb_clf)

Gradient Boosting Accuracy: 0.7993

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.78      0.79       863
           1       0.79      0.82      0.81       876

    accuracy                           0.80      1739
   macro avg       0.80      0.80      0.80      1739
weighted avg       0.80      0.80      0.80      1739

Confusion Matrix:
[[669 194]
 [155 721]]


- Adaptive Boosting

In [12]:
ada_clf = AdaBoostClassifier(n_estimators=100, learning_rate=1.0, random_state=42)
ada_clf.fit(X_train_selected, y_train)
evaluate("AdaBoost", ada_clf)

AdaBoost Accuracy: 0.7694

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.81      0.78       863
           1       0.79      0.73      0.76       876

    accuracy                           0.77      1739
   macro avg       0.77      0.77      0.77      1739
weighted avg       0.77      0.77      0.77      1739

Confusion Matrix:
[[696 167]
 [234 642]]


In [13]:
# All the models side by side (same train/test split, same scaled and selected features)
comparison = pd.DataFrame(results).T.sort_values("accuracy", ascending=False)
comparison.round(4)

,accuracy,precision,recall,f1
Gradient Boosting,0.7993,0.7880,0.8231,0.8051
Random Forest,0.7913,0.7866,0.8037,0.7950
Logistic Regression,0.7907,0.7876,0.8002,0.7939
Bagging,0.7861,0.7763,0.8082,0.7919
Pasting,0.7746,0.7630,0.8014,0.7817
AdaBoost,0.7694,0.7936,0.7329,0.7620
Decision Tree (single),0.7435,0.7137,0.8196,0.7630


Which model is the best and why?

**Conclusion: Gradient Boosting is the best model, but only by a small margin.**

It has the highest accuracy (79.93%) and the highest F1 (0.805), and it also finds the most transported passengers (recall 82.31%). It works well because boosting builds shallow trees one after another, and each new tree focuses on the mistakes of the previous ones, so it reduces the bias of the model, while the small learning rate (0.1) and the shallow trees keep overfitting under control.

What the comparison shows:

- **Ensembles clearly beat a single decision tree.** A single unrestricted tree reaches 74.35% because it overfits. Bagging (78.61%), Random Forest (79.13%) and Gradient Boosting (79.93%) gain between 4 and 6 percentage points by combining many trees.
- **Random Forest is a very close second** (79.13%). It is Bagging plus a random choice of features at every split, which makes the trees less similar and gives a small extra gain over Bagging.
- **Bagging did a little better than Pasting** (78.61% vs 77.46%): sampling with replacement gives more varied trees than sampling without replacement.
- **AdaBoost is the weakest ensemble** (76.94%) and misses the most transported passengers (recall 73.29%); with its default weak learners and a learning rate of 1.0 it is more sensitive to noise.
- **The ensembles barely improve on Logistic Regression** (79.07%), the simple reference model used here because the results of the previous lab are not part of this notebook. Gradient Boosting is only 0.86 points above it. With the same 10 selected features, the limit seems to be the information in the features, not the complexity of the model, so better feature engineering would probably help more than more complex models.

Caution: the test set has 1,739 passengers, so one percentage point is only about 17 passengers. The differences between the top models (Gradient Boosting, Random Forest, Logistic Regression) are within the normal noise of a single train/test split. To pick between them with confidence, use cross-validation and tune the hyperparameters.